# 2. Multi-Agent Customer Support: Tools, Guardrails and Memory

**Learning objectives**
- Build a support agent and a quality-review agent.
- Give an agent a tool to retrieve approved information.
- Validate an output with a task guardrail.
- Understand the purpose and configuration considerations of memory.

This notebook uses a small local FAQ tool so the exercise does not depend on a third-party search API.

# Setup: CrewAI + Google Gemini (Colab)

**Before running:** add a Gemini API key to Google Colab → Secrets with the name `GOOGLE_API_KEY` and enable notebook access to it. A Serper key is optional and is only needed for live web search in the relevant exercises.

These notebooks use the current CrewAI Python API style. The exact package versions are intentionally not pinned to an old 0.x release; after a successful lab setup, freeze the working environment for repeatability.

In [ ]:
%pip -q install -U "crewai[google-genai]" crewai-tools

In [ ]:
import os
from google.colab import userdata
from crewai import Agent, Task, Crew, Process, LLM

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Add GOOGLE_API_KEY in Colab Secrets and enable notebook access.")

llm = LLM(
    model="gemini/gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
    temperature=0.2
)
print("LLM configured.")

## Step 1 — Create a small knowledge source and a custom tool

This is a deliberately small teaching knowledge base. In a real system, replace it with approved documentation and retrieval with citations.

In [ ]:
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

FAQ = {
    "memory": (
        "CrewAI memory can be enabled at the crew level. Memory may retain "
        "useful information across task executions depending on the configured "
        "storage and embedding provider. Review the current CrewAI memory docs "
        "and configure the provider explicitly for production."
    ),
    "sequential": (
        "A sequential crew executes tasks in the order they are listed. "
        "The output of earlier tasks can be passed as context to later tasks."
    ),
    "tools": (
        "Tools are callable capabilities that let an agent interact with "
        "external systems or perform operations beyond text generation."
    )
}

class FAQInput(BaseModel):
    query: str = Field(..., description="The topic or question to look up in the FAQ.")

class CollegeSupportFAQTool(BaseTool):
    name: str = "Approved Support FAQ"
    description: str = (
        "Search a small approved FAQ about CrewAI concepts. "
        "Use it before answering questions about memory, sequential tasks or tools."
    )
    args_schema: type[BaseModel] = FAQInput

    def _run(self, query: str) -> str:
        q = query.lower()
        matches = [value for key, value in FAQ.items() if key in q]
        if matches:
            return "\n".join(matches)
        return (
            "No matching FAQ entry was found. State that the local knowledge "
            "base does not contain the answer; do not invent a policy."
        )

faq_tool = CollegeSupportFAQTool()
print(faq_tool.run(query="How does memory work?"))

## Step 2 — Define the agents

The support agent has access to the FAQ tool. The QA agent checks the response and can request corrections through its own task instructions.

In [ ]:
support_agent = Agent(
    role="Customer Support Specialist",
    goal="Answer the customer's question accurately using the approved FAQ",
    backstory=(
        "You are a patient support specialist. Use the approved FAQ tool "
        "for product guidance. If the answer is not in the FAQ, say so clearly "
        "and suggest consulting official documentation."
    ),
    tools=[faq_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=5
)

qa_agent = Agent(
    role="Support Quality Reviewer",
    goal="Check that the proposed answer is accurate, complete and transparent",
    backstory=(
        "You review support answers. You remove unsupported claims, check "
        "that uncertainty is stated and ensure that all parts of the question "
        "are addressed."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
    max_iter=4
)

## Step 3 — Create tasks and a guardrail

A guardrail is a validation function. When it rejects an output, CrewAI can send the reason back to the agent for a retry.

In [ ]:
support_task = Task(
    description=(
        "Customer: {customer}\nContact: {person}\nQuestion: {inquiry}\n\n"
        "Use the Approved Support FAQ tool. Answer only what is supported. "
        "If the FAQ does not contain the answer, say what is missing."
    ),
    expected_output="A concise, friendly answer with a note about the information source or limitation.",
    agent=support_agent
)

def support_answer_guardrail(output):
    text = output.raw.strip()
    if len(text) < 40:
        return (False, "The answer is too brief. Address the customer's question in complete sentences.")
    if "invent" in text.lower():
        return (False, "Rewrite the answer without unsupported assumptions.")
    return (True, text)

qa_task = Task(
    description=(
        "Review the draft response for {customer}. Confirm that it answers "
        "the question, uses only the available FAQ information, and clearly "
        "states any limitation. Return the corrected final answer."
    ),
    expected_output="A final customer-ready answer, with unsupported claims removed.",
    agent=qa_agent,
    guardrail=support_answer_guardrail,
    guardrail_max_retries=2
)

## Step 4 — Assemble and run the Crew

In [ ]:
support_crew = Crew(
    agents=[support_agent, qa_agent],
    tasks=[support_task, qa_task],
    process=Process.sequential,
    verbose=True
)

inputs = {
    "customer": "CampusHelp (fictional customer)",
    "person": "Aarav",
    "inquiry": "How can I enable memory for a CrewAI crew, and what should I consider?"
}

support_result = support_crew.kickoff(inputs=inputs)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(support_result.raw))

## Step 5 — Memory: understand before enabling

CrewAI memory is not the same as simply keeping the current prompt in context. A memory-enabled crew can store and retrieve information across runs, depending on its configuration.

The following is an **optional demonstration**. Run it after confirming the embedding provider and storage configuration for your environment. Memory may make additional model/API calls and can persist data locally.

In [ ]:
# OPTIONAL: Enable built-in CrewAI memory.
# Check the current CrewAI Memory documentation and configure the embedder
# explicitly if your environment does not have a suitable default provider.

memory_demo_crew = Crew(
    agents=[support_agent],
    tasks=[Task(
        description=(
            "The customer {customer} says: {message}. "
            "Respond briefly and remember only the non-sensitive preference "
            "that the customer has explicitly provided."
        ),
        expected_output="A short acknowledgement.",
        agent=support_agent
    )],
    process=Process.sequential,
    memory=True,
    verbose=True
)

# First run: state a harmless preference.
# Uncomment to execute:
# memory_demo_crew.kickoff(inputs={
#     "customer": "CampusHelp",
#     "message": "For future answers, please use concise bullet points."
# })

# Second run: ask whether the preference can be applied.
# Uncomment to execute after the first run:
# memory_demo_crew.kickoff(inputs={
#     "customer": "CampusHelp",
#     "message": "Explain sequential task execution."
# })

## Try it yourself
1. Ask about `sequential`, `tools` and an unknown topic.
2. Modify the FAQ tool to add a new approved entry.
3. Deliberately make the guardrail fail and observe the retry.
4. Compare the answer with and without the QA task.

**Safety note:** Never store passwords, API keys, payment details or sensitive personal information in agent memory. Explain what is retained and provide a way to clear it.